In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_68_try_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 69 ###

def count_then_return_percent_81(df, col):
    # GPU‐accelerated count and percentage calculation
    counts = df[col].value_counts(dropna=False)
    return counts.mul(100).div(counts.sum()).round(1)

# Define the question strings
tq_2022 = 'Approximately how many times have you used a TPU (tensor processing unit)?'
tq_2019 = 'Have you ever used a TPU (tensor processing unit)?'
question_of_interest = tq_2022

# Standardize and rename the 2019 responses in one GPU call
responses_df_2019_cell10 = responses_df_2019_cell10.rename(columns={tq_2019: tq_2022})
responses_df_2019_cell10[tq_2022] = responses_df_2019_cell10[tq_2022].replace({
    '6-24 times': '6-25 times',
    '> 25 times': 'More than 25 times'
})

# Aggregate for each year into a list
df_list = [
    (responses_df_2019_cell10, '2019'),
    (responses_df_2020, '2020'),
    (responses_df_2021, '2021'),
    (responses_df_2022_cell10, '2022')
]
dfs = []
for df, year in df_list:
    tmp = count_then_return_percent_81(df, question_of_interest)
    tmp = tmp.sort_index().to_frame('percentage')
    tmp['year'] = year
    dfs.append(tmp)

# Combine and rename index to the x‐axis title
x_axis_title_cell81 = 'Frequency of TPU usage'
df_combined = pd.concat(dfs)
df_combined = df_combined.reset_index().rename(columns={'index': x_axis_title_cell81})

# Final sorting and inspection
df_combined_cell81 = df_combined.sort_values(by=['year', 'percentage'], ascending=True)
df_combined_cell81.info()